# Data Acquisition and Preparation

This notebook collects and prepares source data used in the project. The cells document the download, inspection, transformation, and export steps. The final modeling notebooks use the cleaned file in `data/dataset_final/full_data.csv`, so rerunning this notebook is only necessary when the source data or cleaning rules change.

In [2]:
"""
Data Collection Script — Ghana Fuel Price Forecasting Project
Collects: Brent Crude Oil, GHS/USD Exchange Rate, CPI (Inflation)
Sources:  EIA API, Frankfurter API, World Bank API
Authors:  [Your Group Names]
Date:     2026
"""

import requests
import pandas as pd
from datetime import datetime
import time

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
START_DATE = "2013-01-01"   # Change to match your study period
END_DATE   = "2026-06-04"   # Change to match your study period
EIA_API_KEY = "5L8SAT2zgbtGXdX2E3gadXuj7cpQPTPhVJBbyMOz"  # Register free at https://www.eia.gov/opendata/


# ─────────────────────────────────────────────
# 1. BRENT CRUDE OIL PRICES — EIA API
# ─────────────────────────────────────────────
def get_brent_crude(api_key, start, end):
    print("Fetching Brent Crude Oil prices from EIA...")
    url = (
        f"https://api.eia.gov/v2/petroleum/pri/spt/data/"
        f"?api_key={api_key}"
        f"&frequency=monthly"
        f"&data[0]=value"
        f"&facets[series][]=RBRTE"
        f"&start={start[:7]}"
        f"&end={end[:7]}"
        f"&sort[0][column]=period"
        f"&sort[0][direction]=asc"
        f"&offset=0&length=5000"
    )
    response = requests.get(url)
    if response.status_code != 200:
        print(f"  ERROR: {response.status_code} - {response.text}")
        return None

    data = response.json()
    records = data.get("response", {}).get("data", [])
    if not records:
        print("  No data returned. Check your API key or date range.")
        return None

    df = pd.DataFrame(records)[["period", "value"]]
    df.columns = ["date", "brent_crude_usd"]
    df["date"] = pd.to_datetime(df["date"])
    df["brent_crude_usd"] = pd.to_numeric(df["brent_crude_usd"], errors="coerce")
    print(f"  Retrieved {len(df)} monthly observations.")
    return df


# ─────────────────────────────────────────────
# 2. GHS/USD EXCHANGE RATE — Frankfurter API
# ─────────────────────────────────────────────
def get_exchange_rate(start, end):
    print("Fetching GHS/USD exchange rate from Frankfurter...")
    records = []
    # Frankfurter returns daily — we will resample to monthly
    url = f"https://api.frankfurter.app/{start}..{end}?from=USD&to=GHS"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"  ERROR: {response.status_code} - {response.text}")
        return None

    data = response.json()
    rates = data.get("rates", {})

    for date_str, currency_dict in rates.items():
        ghs_rate = currency_dict.get("GHS")
        if ghs_rate:
            records.append({"date": date_str, "exchange_rate_ghs_usd": ghs_rate})

    if not records:
        print("  No data returned.")
        return None

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")

    # Resample to monthly — take last observation of each month
    df_monthly = df.resample("MS").last().reset_index()
    print(f"  Retrieved {len(df_monthly)} monthly observations.")
    return df_monthly


# ─────────────────────────────────────────────
# 3. CPI / INFLATION — World Bank API
# ─────────────────────────────────────────────
def get_cpi_worldbank(start, end):
    print("Fetching CPI data from World Bank...")
    start_year = int(start[:4])
    end_year   = int(end[:4])

    url = (
        f"https://api.worldbank.org/v2/country/GH/indicator/FP.CPI.TOTL"
        f"?date={start_year}:{end_year}&format=json&per_page=100"
    )
    response = requests.get(url)
    if response.status_code != 200:
        print(f"  ERROR: {response.status_code}")
        return None

    data = response.json()
    if len(data) < 2 or not data[1]:
        print("  No data returned.")
        return None

    records = []
    for entry in data[1]:
        if entry.get("value") is not None:
            records.append({
                "date": pd.to_datetime(f"{entry['date']}-01-01"),
                "cpi": entry["value"]
            })

    if not records:
        return None

    df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)

    # World Bank CPI is annual — interpolate to monthly
    df = df.set_index("date")
    df_monthly = df.resample("MS").interpolate(method="linear").reset_index()
    df_monthly = df_monthly[
        (df_monthly["date"] >= START_DATE) &
        (df_monthly["date"] <= END_DATE)
    ]
    print(f"  Retrieved {len(df_monthly)} monthly observations (interpolated from annual).")
    print("  NOTE: CPI is annual from World Bank — interpolated to monthly.")
    print("  Consider replacing with GSS monthly CPI reports for higher accuracy.")
    return df_monthly


# ─────────────────────────────────────────────
# 4. MERGE ALL DATASETS
# ─────────────────────────────────────────────
def merge_datasets(brent_df, fx_df, cpi_df):
    print("\nMerging all datasets...")

    # Standardise date column to month start
    for df in [brent_df, fx_df, cpi_df]:
        df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()

    merged = brent_df.merge(fx_df, on="date", how="outer")
    merged = merged.merge(cpi_df, on="date", how="outer")
    merged = merged.sort_values("date").reset_index(drop=True)

    # Filter to study period
    merged = merged[
        (merged["date"] >= START_DATE) &
        (merged["date"] <= END_DATE)
    ]

    print(f"  Final dataset: {len(merged)} rows x {len(merged.columns)} columns")
    print(f"  Date range: {merged['date'].min()} to {merged['date'].max()}")
    print(f"  Missing values:\n{merged.isnull().sum()}")
    return merged


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("  Ghana Fuel Price Project — Data Collection")
    print("=" * 55)

    brent_df = get_brent_crude(EIA_API_KEY, START_DATE, END_DATE)
    time.sleep(1)

    fx_df = get_exchange_rate(START_DATE, END_DATE)
    time.sleep(1)

    cpi_df = get_cpi_worldbank(START_DATE, END_DATE)

    if brent_df is not None and fx_df is not None and cpi_df is not None:
        final_df = merge_datasets(brent_df, fx_df, cpi_df)
        output_path = "external_variables.csv"
        final_df.to_csv(output_path, index=False)
        print(f"\n✓ Data saved to: {output_path}")
        print("\nFirst 5 rows:")
        print(final_df.head())
    else:
        print("\n✗ One or more datasets failed. Check errors above.")

    print("\nNOTE: Fuel prices (petrol, diesel, LPG) must be")
    print("added manually from NPA. Add as columns to the CSV.")

  Ghana Fuel Price Project — Data Collection
Fetching Brent Crude Oil prices from EIA...
  Retrieved 161 monthly observations.
Fetching GHS/USD exchange rate from Frankfurter...
  ERROR: 404 - {"message":"not found"}
Fetching CPI data from World Bank...
  Retrieved 133 monthly observations (interpolated from annual).
  NOTE: CPI is annual from World Bank — interpolated to monthly.
  Consider replacing with GSS monthly CPI reports for higher accuracy.

✗ One or more datasets failed. Check errors above.

NOTE: Fuel prices (petrol, diesel, LPG) must be
added manually from NPA. Add as columns to the CSV.


In [3]:
brent_df.head()

,date,brent_crude_usd
0,2013-01-01,112.96
1,2013-02-01,116.05
2,2013-03-01,108.47
3,2013-04-01,102.25
4,2013-05-01,102.56


In [4]:
brent_df.tail()

,date,brent_crude_usd
156,2026-01-01,66.60
157,2026-02-01,70.89
158,2026-03-01,103.13
159,2026-04-01,117.29
160,2026-05-01,107.14


In [7]:
brent_df.to_csv("external_variables.csv", index=False)

In [8]:
cpi_df.to_csv("cpi_data.csv", index=False)

In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# --- Load and prepare (same as your pipeline) ---
df = pd.read_csv("../data/dataset_final/full_data.csv")
df['date'] = pd.to_datetime(df['month'], format='%YM%m')
df.set_index('date', inplace=True)
df = df.drop(columns='month')
df.index.freq = 'MS'

print("Shape:", df.shape)
print(df.isnull().sum())

arima_cols = ['diesel_price_(GHC/litre)', 'petrol_price_ghs/litre', 'lpg_ghs/kg',
              'brent_price_usd_per_barrel', 'usd/ghs_rate']
var_cols = ['diesel_price_(GHC/litre)', 'petrol_price_ghs/litre',
            'brent_price_usd_per_barrel', 'usd/ghs_rate', 'consumer_price_index']

df_diff1 = df[arima_cols].diff().dropna()
df_diff2 = df[var_cols].diff().diff().dropna()

def plot_acf_pacf(series, name, lags=24, figsize=(12, 4)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    plot_acf(series, lags=lags, ax=axes[0])
    axes[0].set_title(f'ACF: {name}')
    plot_pacf(series, lags=lags, ax=axes[1], method='ywm')
    axes[1].set_title(f'PACF: {name}')
    plt.tight_layout()
    safe = name.replace("/", "_").replace(" ", "_").replace("(","").replace(")","")
    plt.savefig(f'../documentation/real_acf_pacf_{safe}.png', dpi=110)
    plt.close()
    print(f"Saved: {name} (n={len(series)})")

print("\n=== ARIMA/ARIMAX set (d=1) ===")
for col in arima_cols:
    plot_acf_pacf(df_diff1[col], col, lags=24)

print("\n=== VAR set (d=2) ===")
for col in var_cols:
    plot_acf_pacf(df_diff2[col], col, lags=24)

Shape: (132, 6)
diesel_price_(GHC/litre)      0
lpg_ghs/kg                    0
petrol_price_ghs/litre        0
brent_price_usd_per_barrel    0
usd/ghs_rate                  0
consumer_price_index          0
dtype: int64

=== ARIMA/ARIMAX set (d=1) ===
Saved: diesel_price_(GHC/litre) (n=131)
Saved: petrol_price_ghs/litre (n=131)
Saved: lpg_ghs/kg (n=131)
Saved: brent_price_usd_per_barrel (n=131)
Saved: usd/ghs_rate (n=131)

=== VAR set (d=2) ===
Saved: diesel_price_(GHC/litre) (n=130)
Saved: petrol_price_ghs/litre (n=130)
Saved: brent_price_usd_per_barrel (n=130)
Saved: usd/ghs_rate (n=130)
Saved: consumer_price_index (n=130)
